<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Gradient_Boosting_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ----------------------------------------------------------
# Feature Engineering
# ----------------------------------------------------------

from ta.momentum import RSIIndicator
import numpy as np

# Daily Return
df["Return"] = df["Close"].pct_change()

# Lagged Returns
df["Lag1"] = df["Return"].shift(1)
df["Lag2"] = df["Return"].shift(2)
df["Lag3"] = df["Return"].shift(3)

# Moving Averages
df["MA5"] = df["Close"].rolling(5).mean()
df["MA10"] = df["Close"].rolling(10).mean()

# Volatility
df["Volatility20"] = df["Return"].rolling(20).std()

# RSI (using ta library)
df["RSI"] = RSIIndicator(close=df["Close"], window=14).rsi()

# Target (Next-Day Return)
df["Target"] = df["Return"].shift(-1)

# Remove infinities
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop missing rows
df.dropna(inplace=True)

# ----------------------------------------------------------
# Features
# ----------------------------------------------------------

features = [
    "Lag1",
    "Lag2",
    "Lag3",
    "MA5",
    "MA10",
    "RSI",
    "Volatility20"
]

X = df[features]
y = df["Target"]

# ----------------------------------------------------------
# Train/Test Split
# ----------------------------------------------------------

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# Final safety check
print("NaN values:", X_train.isna().sum().sum())
print("Infinite values:", np.isinf(X_train).sum().sum())

NaN values: 0
Infinite values: 0


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr.fit(X_train, y_train)
pred_gbr = gbr.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate(name, y_true, pred):

    print("="*50)
    print(name)
    print("="*50)

    print("R²:", round(r2_score(y_true, pred), 4))
    print("MSE:", round(mean_squared_error(y_true, pred), 8))
    print("MAE:", round(mean_absolute_error(y_true, pred), 6))

    direction = (np.sign(pred) == np.sign(y_true)).mean() * 100
    print("Directional Accuracy:", round(direction, 2), "%")

evaluate("Linear Regression", y_test, pred_lr)
evaluate("Gradient Boosting", y_test, pred_gbr)

Linear Regression
R²: -0.0059
MSE: 7.192e-05
MAE: 0.006232
Directional Accuracy: 51.98 %
Gradient Boosting
R²: -0.9895
MSE: 0.00014225
MAE: 0.009162
Directional Accuracy: 51.15 %


In [ ]:
!pip -q install ta

from ta.momentum import RSIIndicator

  Preparing metadata (setup.py) ... done
